# Cevap katmanı — gerçek dil modeliyle ilk çalıştırma

**Ne yapar:** Arşivi Qdrant'a yükler, `pipeline.cevapla()` fonksiyonunu
gerçek bir dil modeliyle çalıştırır ve atıflı cevaplar üretir.

**Neden Colab:** yerel makinede GPU yok. 7 milyarlık bir model CPU'da
saniyede birkaç token üretir; bir cevap dakikayı bulur.

**Önce yapılacak:** `Runtime > Change runtime type > T4 GPU`.

**Sıra önemli:** 6. hücre modeli yüklemeden, sahte arka uçla boruları test
eder. Orada bir sorun varsa modeli yüklemek için beş dakika harcamaya gerek
kalmaz. Model yüklemesi tek pahalı adım, o yüzden en sona bırakıldı.

In [ ]:
# 1) GPU var mı? Beklenen: 'Tesla T4' satırı ve cuda: True.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('cuda:', torch.cuda.is_available())

In [ ]:
# 2) Kurulum. 2-3 dakika.
#    bitsandbytes dört bitlik yükleme için: 7B model yarım hassasiyette ~15 GB
#    tutuyor ve T4'ün 16 GB'ına aktivasyonlarla sığmıyor. Dört bitte ~5 GB.
!pip install -q sentence-transformers qdrant-client transformers accelerate bitsandbytes
import sentence_transformers, transformers
print('sentence-transformers', sentence_transformers.__version__)
print('transformers', transformers.__version__)

## Yüklenecek dosyalar

Önce yerel makinede şunu çalıştır — dosyaları tek klasöre toplar ve
güncelliklerini doğrular:

```
_env\parse\Scripts\python.exe _scripts\colab_paketle.py
```

Sonra aşağıdaki hücrede **`C:\Users\Tunahan\Desktop\myeloma\1`** klasöründeki
**on üç dosyanın hepsini** seç (Ctrl+A).

Qdrant koleksiyonunu yüklemiyoruz, Colab'da yeniden kuruyoruz: klasör 169 MB,
oysa kaynak dosyalar 87 MB ve kurulum 70 saniye sürüyor.

**Hücre sırası:** 1-7 kurulum ve doğrulama, 8 dört soruluk demo, 10 ise
78+15 soruluk Level 3 ölçümü (40-60 dakika). Demo hızlı geri bildirim için;
ölçüm rapora girecek rakamlar için. İkisi aynı `cevapla()` fonksiyonunu
çağırıyor, yani gösterdiğimiz sistemle ölçtüğümüz sistem aynı.

In [ ]:
# 3) Yükleme + klasör yapısı. Eski _scripts siliniyor ki önceki oturumdan
#    kalan bir dosya sessizce çalışmasın - bu hata bir kez oldu.
import os, shutil
from google.colab import files

KOK = '/content/myeloma'
YER = {
    'chunks.json':           '_work/chunks',
    'golden_queries.json':   '_work/eval',
    'negative_queries.json': '_work/eval',
    'negative_queries_v2.json': '_work/eval',
    'negatif_taze.py':       '_scripts',
    'BAAI_bge-m3.npy':       '_work/embeddings',
    'BAAI_bge-m3.meta.json': '_work/embeddings',
    'pipeline.py':           '_scripts',
    'answer_eval.py':        '_scripts',
    'qdrant_index.py':       '_scripts',
    'query_processing.py':   '_scripts',
    'retrieval_eval.py':     '_scripts',
    'embed_chunks.py':       '_scripts',
    'chunk.py':              '_scripts',
    'plog.py':               '_scripts',
}
shutil.rmtree(os.path.join(KOK, '_scripts'), ignore_errors=True)
for d in set(YER.values()) | {'_logs'}:
    os.makedirs(os.path.join(KOK, d), exist_ok=True)

for ad in files.upload():
    temiz = ad.replace(' (1)', '').replace(' (2)', '')
    if temiz not in YER:
        print(f'  atlandi: {ad}')
        continue
    shutil.move(ad, os.path.join(KOK, YER[temiz], temiz))

eksik = [a for a, d in YER.items()
         if not os.path.exists(os.path.join(KOK, d, a))]
print('eksik:', eksik or 'yok')
assert not eksik, 'eksik dosya var - hücreyi tekrar çalıştırıp hepsini seç'

In [ ]:
# 4) Qdrant koleksiyonunu kur. ~70 saniye, 14.532 nokta.
#    kur() içinde 'vektör sayısı = chunk sayısı' kontrolü var; tutmazsa durur.
#    Tutmasaydı sistem hata vermeden yanlış cevap üretirdi.
%env HF_HOME=/content/hf_cache
import sys
sys.path.insert(0, f'{KOK}/_scripts')
os.chdir(KOK)

import qdrant_index as QI
QI.kur(yenile=True)

In [ ]:
# 5) Taşıma doğru mu? 20 rastgele sorguda Qdrant ile numpy aynı ilk 10'u
#    vermeli. Beklenen: '20/20'.
QI.dogrula()

In [ ]:
# 6) MODEL YÜKLEMEDEN boru testi. Sahte arka uç cevap kalitesi üretmiyor;
#    arama, tekrar ayıklama, künye ve reddetme çalışıyor mu ona bakıyor.
#    Beklenen: ilk soruda karfilzomib kaynakları, ikincisinde reddedildi=True.
import pipeline as P

istemci = QI.ac()
for s in ['Karfilzomib kardiyak toksisite açısından hangi uyarıları taşıyor?',
          'Kedi maması hangi markadan alınmalı?']:
    r = P.cevapla(s, istemci=istemci)
    print('=' * 70)
    print(s)
    print('  güven:', r['guven']['seviye'], '| skor', r['guven']['arama_skoru'],
          '| reddedildi:', r['reddedildi'])
    for x in r['kaynaklar'][:3]:
        print('   ', P.kunye(x))

In [ ]:
# 7) Gerçek model. İlk çalıştırmada ~5 GB iner, 4-6 dakika.
#    Qwen2.5 seçildi: bu boyut sınıfında Türkçesi iyi ve yönerge takibi güçlü -
#    bizim istemimiz katı kurallara dayanıyor, model kuralı tutmazsa atıf da
#    reddetme de çalışmaz.
#    Bellek yetmezse 'Qwen/Qwen2.5-3B-Instruct' ile dene.
import time
t0 = time.time()
llm = P.llm_kur('transformers', 'Qwen/Qwen2.5-7B-Instruct')
print(f'model yüklendi {time.time()-t0:.0f}sn')

In [ ]:
# 8) Demo senaryoları. Dördü de farklı bir şeyi sınıyor.
#    k=4: T4'te sekiz parçalık bağlam belleği taşırdı. Dikkat hesabının belleği
#    dizi uzunluğunun karesiyle büyüyor; dört parça ~3.000 token.
#    Yine taşarsa k=2 yap; o da taşarsa 7. hücrede 3B modele in.
import time, gc, torch

SORULAR = [
    # (a) Doz sorusu - sayısal değerler kaynaktaki gibi aktarılmalı
    'Karfilzomib hangi dozda ve hangi günlerde uygulanır?',
    # (b) İlaç karışması - teklistamab sorulduğunda elranatamab gelmemeli,
    #     ve model bu kez ATIF vermeli (ilk koşuda hiç atıf yoktu)
    'Teklistamab tedavisinde sitokin salınım sendromu için basamaklı doz şeması nedir?',
    # (c) Geri ödeme - SUT katmanı devreye girmeli
    'Lenalidomid SUT kapsamında hangi koşullarda ödenir?',
    # (d) Arşivde olmayan - sistem uydurmak yerine durmalı.
    #     İlk koşuda skor 0,565 ile eşiği kıl payı geçip cevap uydurmuştu;
    #     artık ikinci kapı (dayanaklılık < 0,40) bunu durdurmalı.
    'Akut apandisit ameliyatı sonrası antibiyotik profilaksisi nasıl verilir?',
]

sonuclar = []
for s in SORULAR:
    gc.collect()
    torch.cuda.empty_cache()
    t0 = time.time()
    r = P.cevapla(s, llm=llm, istemci=istemci, k=4)
    r['saniye'] = round(time.time() - t0, 1)
    sonuclar.append(r)
    P.yazdir(r)
    if r['guven'].get('dayanak_kapisi'):
        print('  KAPI:', r['guven']['dayanak_kapisi'])
        print('  gizlenen cevap:', r['guven']['gizlenen_cevap'][:200])
    print(f"({r['saniye']} sn)")
    print('=' * 70)

In [ ]:
# 9) Sonuçları indir. Yerelde inceleyip Level 3 ölçümüne temel yapacağız.
import json
for r in sonuclar:
    r.pop('analiz', None)
json.dump(sonuclar, open('/content/cevap_demo.json', 'w', encoding='utf-8'),
          ensure_ascii=False, indent=1)
from google.colab import files
files.download('/content/cevap_demo.json')

In [ ]:
# 10) LEVEL 3 ÖLÇÜMÜ - 78 pozitif + 15 negatif soru. T4'te 40-60 dakika.
#     Dört demo sorusu hikâye anlatır, 93 soru rapor yazdırır.
#
#     Kurgu: eşik YALNIZCA soruların yarısına (A) bakılarak seçiliyor,
#     raporlanan rakam diğer yarıdan (B) geliyor. Eşiği bütün kümeye bakarak
#     seçmek, sınavı gördükten sonra geçme notu belirlemek olurdu.
#
#     Dil modeli BİR KEZ koşuyor, kapılar kapalı; eşikler sonradan ham
#     kayıtlara uygulanıyor. Yoksa her eşik denemesi 40 dakika sürerdi.
#
#     NOT: alt süreç olarak çalıştırmıyoruz - o zaman model ikinci kez yüklenir
#     ve bu hücredeki 'llm' GPU belleğini tuttuğu için taşar. Yüklü modeli
#     yeniden kullanıyoruz.
import json, answer_eval as AE

poz = json.load(open(f'{KOK}/_work/eval/golden_queries.json', encoding='utf-8'))
neg = json.load(open(f'{KOK}/_work/eval/negative_queries.json', encoding='utf-8'))
chunk_haritasi = {c['chunk_id']: c for c in
                  json.load(open(f'{KOK}/_work/chunks/chunks.json',
                                 encoding='utf-8'))}
print(f'{len(poz)} pozitif, {len(neg)} negatif soru')

kp = AE.ham_kos(llm, poz, istemci, chunk_haritasi, 4, pozitif=True)
kn = AE.ham_kos(llm, neg, istemci, chunk_haritasi, 4, pozitif=False)

pa, pb = AE.bol(kp)
na, nb = AE.bol(kn)
arama, dayanak, secim = AE.esik_sec(pa, na)
olcum = AE.puanla(pb, nb, arama, dayanak)

print(f'\nESIK (yalnizca A kumesinden secildi): '
      f'arama {arama}, dayanaklilik {dayanak}')
for ad, s in (('SECIM  (A - esigin gordugu)', secim),
              ('OLCUM  (B - esigin gormedigi)', olcum)):
    print(f"\n{ad}   pozitif {s['pozitif']}, negatif {s['negatif']}")
    print(f"  cevaplanan     {s['cevaplanan']:3d} / {s['pozitif']}")
    print(f"  hedefi tutan   {s['hedef_tutan']:3d} / {s['pozitif']}")
    print(f"  haksiz red     {s['haksiz_red']:3d}")
    print(f"  ort dayanak    {s['ort_dayanaklilik']}")
    print(f"  UYDURMA GECTI  {s['uydurma_gecti']:3d} / {s['negatif']}"
          f"  {s['uydurma_idler'] or ''}")

json.dump({'llm': 'transformers', 'model': 'Qwen/Qwen2.5-7B-Instruct', 'k': 4,
           'esik': {'arama': arama, 'dayanaklilik': dayanak},
           'secim_kumesi': secim, 'olcum_kumesi': olcum,
           'kayitlar': {'pozitif': kp, 'negatif': kn}},
          open('/content/level3_eval.json', 'w', encoding='utf-8'),
          indent=1, ensure_ascii=False)

from google.colab import files
files.download('/content/level3_eval.json')

## Bittikten sonra

İnen `cevap_demo.json` dosyasını `_logs\` içine koy ve bana haber ver.
Ekrandaki cevapları da yapıştır.

Bu ikinci koşu. İlk koşuda dört sorun çıktı ve dördü de düzeltildi; şimdi
düzeltmelerin tuttuğuna bakıyoruz:

**Reddetme çalışıyor mu?** (d) apandisit sorusu. İlk koşuda skor 0,565 ile
0,55 eşiğini kıl payı geçti ve model kendi bilgisinden cevap uydurdu. Artık
ikinci bir kapı var: cevap üretildikten sonra dayanaklılığı ölçülüyor, 0,40'ın
altındaysa gösterilmiyor. Beklenen çıktı `reddedildi: True` ve `KAPI:` satırı.

**Atıf çıkıyor mu?** (b) teklistamab sorusu. İlk koşuda model doğru doz
şemasını verdi ama hiçbir atıf yapmadı, yani değerlerin kaynaktan mı kendi
belleğinden mi geldiği ayırt edilemedi. İsteme "her cümleye ve her maddeye
atıf" kuralı eklendi.

**Doz sadakati.** (a) karfilzomib. Sayısal değerler kaynaktaki gibi mi, yoksa
model yuvarlamış ya da kendi cümlesiyle yeniden mi yazmış?

**Hız.** Soru başına kaç saniye. İlk koşuda 12-51 saniyeydi.

Bir de: `gizlenen cevap` satırı çıkarsa onu da yapıştır. Kapının neyi
durdurduğunu görmek, kapının çalıştığını görmek kadar önemli.

## 11) Taze negatif olcumu - alan kapisi (GPU gerektirmez)

Bu hucre icin yalnizca 1-5. hucreler gerekli; dil modeli yuklenmiyor.
Ilk negatif kumede kalan tek uydurma (N14, "KOAH alevlenmesinde sistemik
steroid") sinif sozlugundeki ciplak "steroid" kelimesinden geliyordu ve
duzeltildi. Ama N14 olcum yarisindaydi: duzeltmeden sonraki rakam, sinavi
gordukten sonra verilmis nottur.

Bu yuzden 15 taze negatif yazildi (N16-N30, agirlikla hematoloji - AML, ITP,
KML, Hodgkin, talasemi) ve esikler DONDURULDU. Ikisi bilerek tuzak: del(5q)
MDS'de lenalidomid ve mantle hucreli lenfomada bortezomib. Ilac arsivde var,
endikasyon yok - varlik tanima bu ikisini alan ici sanacak.

Beklenen: arama ve alan kapilarinin durdurdugu soru sayisi yuksek, modele
ulasan sayi dusuk. Modele ulasan sayi UYDURMA UST SINIRIDIR: dordUncu kapi
(dayanaklilik) bir kismini daha eleyebilir, ama kapiya takilan soru kesin
olarak uydurma uretemez cunku model hic cagrilmiyor.

In [ ]:
# 11) Taze negatif olcumu. AYNI SUREC icinde kosuyor: Qdrant yerel modda depo
#     klasorunu kilitliyor, bu yuzden alt surec olarak calistirilamiyor
#     (defterde acik istemci varken 'already accessed by another instance').
#     GPU gerekmiyor; 1-5. hucreler yeterli. ~1 dakika.
import importlib, json, sys
sys.path.insert(0, f'{KOK}/_scripts')
import negatif_taze as NT
importlib.reload(NT)

try:
    istemci
except NameError:
    istemci = QI.ac()

sorular = json.load(open(f'{KOK}/_work/eval/negative_queries_v2.json',
                         encoding='utf-8'))
print(f"{len(sorular)} taze negatif, esikler donduruldu "
      f"(arama {NT.ARAMA_ESIGI}, alan '{NT.ALAN_KURAL}'):")
kayit = NT.olc(sorular, k=4, istemci=istemci)
NT.ozetle(kayit, f'{KOK}/_logs/negatif_taze.json')

from google.colab import files
files.download(f'{KOK}/_logs/negatif_taze.json')